# Train Logistic Regression on Sign Count Concepts

This notebook trains a multivariate logistic-regression classifier on the sample-level TCAV `sign_count` concept vectors.
It uses a speaker-level 80/20 split, performed separately inside each `source_partition`, to reduce leakage without rerunning TCAV on the manifest train split.


In [ ]:
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler


In [ ]:
# ===== Config =====
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')
TCAV_ROOT = PROJECT_ROOT / 'redimnet' / 'tcav' / 'captum_tcav' / 'asvspoof5'
OUTPUT_SUBDIR = 'subset_20spk_20utts_per_spk_one_logistic_head'
OUTPUT_DIR = TCAV_ROOT / 'output' / OUTPUT_SUBDIR

INPUT_CSV = OUTPUT_DIR / 'merged_tcav_sign_count_wide.csv'
MODEL_OUT_DIR = OUTPUT_DIR / 'sign_count_logistic_results'
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

EXCLUDE_A12 = False
TEST_FRACTION = 0.2
RANDOM_SEED = 42
LOGREG_MAX_ITER = 2000
LOGREG_C = 1.0

print('INPUT_CSV =', INPUT_CSV)
print('Exists =', INPUT_CSV.exists())
print('MODEL_OUT_DIR =', MODEL_OUT_DIR)


In [ ]:
df = pd.read_csv(INPUT_CSV)

if EXCLUDE_A12:
    df = df[df['system_id'] != 'A12'].copy()

meta_cols = ['utt_id', 'speaker_id', 'split', 'source_partition', 'system_id', 'target_class', 'binary_label']
feature_cols = [c for c in df.columns if c not in meta_cols]

print('Rows =', len(df))
print('Speakers =', df['speaker_id'].nunique())
print('Systems =', sorted(df['system_id'].unique().tolist()))
print('Feature columns =', len(feature_cols))
display(df.head())


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
train_speakers = []
test_speakers = []

for part in sorted(df['source_partition'].unique()):
    speakers = np.array(sorted(df.loc[df['source_partition'] == part, 'speaker_id'].unique().tolist()), dtype=object)
    rng.shuffle(speakers)
    n_test = max(1, int(round(len(speakers) * TEST_FRACTION)))
    test_part = speakers[:n_test].tolist()
    train_part = speakers[n_test:].tolist()
    train_speakers.extend(train_part)
    test_speakers.extend(test_part)
    print(f'{part}: total speakers={len(speakers)} train={len(train_part)} test={len(test_part)}')

train_speakers = set(train_speakers)
test_speakers = set(test_speakers)

assert train_speakers.isdisjoint(test_speakers), 'Speaker leakage detected'

train_df = df[df['speaker_id'].isin(train_speakers)].copy().reset_index(drop=True)
test_df = df[df['speaker_id'].isin(test_speakers)].copy().reset_index(drop=True)

print('Train rows =', len(train_df))
print('Test rows =', len(test_df))
print('Train class counts:')
print(train_df['binary_label'].value_counts().sort_index())
print('Test class counts:')
print(test_df['binary_label'].value_counts().sort_index())


In [ ]:
X_tr = train_df[feature_cols].to_numpy(dtype=float)
y_tr = train_df['binary_label'].to_numpy(dtype=int)
X_te = test_df[feature_cols].to_numpy(dtype=float)
y_te = test_df['binary_label'].to_numpy(dtype=int)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

clf = LogisticRegression(
    max_iter=LOGREG_MAX_ITER,
    C=LOGREG_C,
    class_weight=None,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
clf.fit(X_tr_s, y_tr)

p_tr = clf.predict_proba(X_tr_s)[:, 1]
p_te = clf.predict_proba(X_te_s)[:, 1]
yhat_tr = (p_tr >= 0.5).astype(int)
yhat_te = (p_te >= 0.5).astype(int)

summary = {
    'train_accuracy': float(accuracy_score(y_tr, yhat_tr)),
    'test_accuracy': float(accuracy_score(y_te, yhat_te)),
    'train_auc': float(roc_auc_score(y_tr, p_tr)),
    'test_auc': float(roc_auc_score(y_te, p_te)),
    'train_confusion_matrix': confusion_matrix(y_tr, yhat_tr).tolist(),
    'test_confusion_matrix': confusion_matrix(y_te, yhat_te).tolist(),
    'train_rows': int(len(train_df)),
    'test_rows': int(len(test_df)),
    'train_speakers': int(train_df['speaker_id'].nunique()),
    'test_speakers': int(test_df['speaker_id'].nunique()),
    'feature_dim': int(len(feature_cols)),
}

print(json.dumps(summary, indent=2))


In [ ]:
summary_df = pd.DataFrame([
    {
        'split': 'train',
        'accuracy': summary['train_accuracy'],
        'auc': summary['train_auc'],
        'rows': summary['train_rows'],
        'speakers': summary['train_speakers'],
    },
    {
        'split': 'test',
        'accuracy': summary['test_accuracy'],
        'auc': summary['test_auc'],
        'rows': summary['test_rows'],
        'speakers': summary['test_speakers'],
    },
])
display(summary_df)

plot_df = summary_df.copy()
plot_df['label'] = plot_df.apply(lambda r: f"{r['split']}\n(n={int(r['rows'])}, spk={int(r['speakers'])})", axis=1)

metrics = [('accuracy', 'Accuracy'), ('auc', 'AUC')]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (metric, title) in zip(axes, metrics):
    vals = plot_df[metric].astype(float).to_numpy()
    labels = plot_df['label'].tolist()
    ymin = max(0.0, vals.min() - 0.02)
    ymax = min(1.0, vals.max() + 0.01)
    if ymax - ymin < 0.02:
        ymin = max(0.0, vals.min() - 0.01)
        ymax = min(1.0, vals.max() + 0.01)
    bars = ax.bar(labels, vals, color=['#4C78A8', '#F58518'], width=0.6)
    ax.set_title(f'Sign Count Logistic {title}')
    ax.set_ylim(ymin, ymax)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
coef_df = pd.DataFrame({
    'concept': feature_cols,
    'coef': clf.coef_[0],
})
coef_df['abs_coef'] = coef_df['coef'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=False).reset_index(drop=True)

print('Intercept:', float(clf.intercept_[0]))
display(coef_df)
display(coef_df.head(10))


In [ ]:
top_df = coef_df.head(12).copy().sort_values('coef')
plt.figure(figsize=(8, 5))
colors = ['#D62728' if v < 0 else '#2CA02C' for v in top_df['coef']]
plt.barh(top_df['concept'], top_df['coef'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Most Influential Sign Count Concepts')
plt.xlabel('Logistic Regression Coefficient')
plt.tight_layout()
plt.show()


In [ ]:
with open(MODEL_OUT_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(MODEL_OUT_DIR / 'logistic_regression.pkl', 'wb') as f:
    pickle.dump(clf, f)
with open(MODEL_OUT_DIR / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

coef_df.to_csv(MODEL_OUT_DIR / 'coefficients.csv', index=False)
train_df.to_csv(MODEL_OUT_DIR / 'train_split.csv', index=False)
test_df.to_csv(MODEL_OUT_DIR / 'test_split.csv', index=False)

pred_train = train_df[meta_cols].copy()
pred_train['p_spoof'] = p_tr
pred_train['y_hat'] = yhat_tr
pred_train['split_role'] = 'train'

pred_test = test_df[meta_cols].copy()
pred_test['p_spoof'] = p_te
pred_test['y_hat'] = yhat_te
pred_test['split_role'] = 'test'

pred_df = pd.concat([pred_train, pred_test], axis=0, ignore_index=True)
pred_df.to_csv(MODEL_OUT_DIR / 'predictions.csv', index=False)

print('Saved outputs to:', MODEL_OUT_DIR)
